# Tutorial 02 — Projects & Data Management with bw2data

Companion explainer: **02_projects_and_data.md**. A tour of the bw2data API:
projects, databases, activities, exchanges, methods — create, query, copy,
delete.

In [1]:
import bw2data as bd
bd.projects.set_current("bw25-tutorials")
BIOSPHERE = next(d for d in bd.databases if "biosphere" in d.lower())
bio = bd.Database(BIOSPHERE)

## Projects

In [2]:
print("current project:", bd.projects.current)
print("project dir:", bd.projects.dir)
print("all projects:", [p.name for p in bd.projects])

current project:

bw25-tutorials

project dir:

C:\Users\derne\AppData\Local\pylca\Brightway3\bw25-tutorials.a86a346f

all projects:

['default', 'bw25-tutorials', 'cs1-bottle', 'cs2-biofuel', 'cs3-cement', 'bw-mcp-test', 'bw-mcp-demo', 'jsonld-lcia-test']

## Databases: two ways to create data
(1) bulk `write`, (2) incremental `new_activity`/`new_exchange`.

In [3]:
co2 = next(f for f in bio
           if f["name"] == "Carbon dioxide, fossil" and f["categories"] == ("air",))

# (1) bulk write
DB = "t02_demo"
if DB in bd.databases:
    del bd.databases[DB]
bd.Database(DB).write({
    (DB, "steel"): {
        "name": "steel production", "unit": "kilogram",
        "exchanges": [
            {"input": (DB, "steel"), "amount": 1.0, "type": "production"},
            {"input": co2.key, "amount": 1.9, "type": "biosphere"},
        ],
    },
})
print("bulk-written db has", len(bd.Database(DB)), "node(s)")

# (2) incremental
DB2 = "t02_demo_incr"
if DB2 in bd.databases:
    del bd.databases[DB2]
db2 = bd.Database(DB2)
db2.register()
act = db2.new_activity(code="alu", name="aluminium production", unit="kilogram")
act.save()
act.new_exchange(input=act, amount=1.0, type="production").save()
act.new_exchange(input=co2, amount=8.2, type="biosphere").save()
db2.process()
print("incremental db has", len(db2), "node(s)")

13:15:39-0400

 [

warning  

] 

Not able to determine geocollections for all datasets. This database is not ready for regionalization.

...

1

node(s)

incremental db has

1

node(s)

## Activities: lookup, metadata, edges, upstream, copy

In [4]:
steel = bd.get_node(database=DB, code="steel")
print("key:", steel.key)
print("name / unit:", steel["name"], "/", steel["unit"])
print("exchanges:")
for e in steel.exchanges():
    print("   ", e["type"], e.amount, "<-", e.input["name"])

# who consumes CO2 upstream? (reverse edges) — here nobody consumes steel yet
print("consumers of steel:", len(list(steel.upstream())))

steel_v2 = steel.copy(code="steel-v2")
steel_v2["name"] = "steel production (EAF)"
steel_v2.save()
print("copied ->", steel_v2)

key:

('t02_demo', 'steel')

name / unit:

steel production

/

...

Carbon dioxide, fossil

consumers of steel:

0

copied ->

'steel production (EAF)' (kilogram, None, None)

## Searching

In [5]:
# full-text search on a database
print("search 'aluminium':", [str(a) for a in db2.search("aluminium")])
# brute-force filter (reliable, index-lag-proof)
print("brute filter:", [a["name"] for a in bd.Database(DB) if "steel" in a["name"]])

search 'aluminium':

["'aluminium production' (kilogram, GLO, None)"]

brute filter:

['steel production (EAF)', 'steel production']

## Exploring biosphere & methods

In [6]:
from collections import Counter
cats = Counter(f["categories"][0] if f["categories"] else "?" for f in bio)
print("biosphere flows by top-level compartment:")
for c, n in cats.most_common(6):
    print(f"   {c:20s} {n}")

co2_variants = [f["name"] for f in bio if "Carbon dioxide" in f["name"]]
print("\nCO2 variants (first 5):", co2_variants[:5])

ipcc = [m for m in bd.methods if "IPCC" in str(m) and "GWP100" in str(m).replace(" ", "")]
print("\nIPCC GWP100 method variants:", len(ipcc))
for m in ipcc[:4]:
    print("   ", m)

biosphere flows by top-level compartment:

   water                1646

   air                  1597

   soil                 757

   natural resource     344

...

18

('IPCC 2013 no LT', 'climate change no LT', 'global warming potential (GWP100) no LT')

('IPCC 2013', 'climate change', 'global warming potential (GWP100)')

('IPCC 2021 no LT', 'climate change no LT', 'global warming potential (GWP100) no LT')

('IPCC 2021 no LT', 'climate change: biogenic no LT', 'global warming potential (GWP100) no LT')

## Cleanup

In [7]:
del bd.databases[DB]
del bd.databases[DB2]
print("remaining databases:", list(bd.databases))

remaining databases:

['ecoinvent-3.10-biosphere', 'smoke_widget', 'smoke_incr', 't03_kettle', 't05_kettle', 't04_kettle_xl', 't06_kettle', 't07_kettle', 't08_param', 't08_sweep', 't09_cups', 't10_sys', 't01_widget']

Next: **03 — building foreground inventories** (linked to real biosphere flows,
verified against a hand calculation).